<a href="https://colab.research.google.com/github/jagtappranav2721-cpu/ML-Internship-FlyRank/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — Content Lifecycle: Growing vs Declining

The paper compares pages that are growing and declining based on their traffic trend. Trend direction is defined using the last 30 days compared with the 30 days before that. The paper reports that growing pages are younger on average than declining pages (185 days vs. 228 days).

**My methodology question:**  
How is the growing-versus-declining label constructed, and does the comparison account for other differences between the two groups? For example, growing and declining pages may differ in search visibility, impressions, content type, or historical performance in addition to content age. Therefore, an observed difference between the groups does not by itself establish that content age caused the difference.

I would treat this finding as an observed relationship between content age and traffic direction unless the methodology provides evidence that other relevant differences were controlled for.

---

### Finding 4 — The Freshness Multiplier

The paper reports that pages updated 31–90 days ago had a 5.43:1 growth-to-decline ratio. It also reports a separate comparison for pages older than 365 days, where the recently refreshed cohort showed a 1.6× health lift and a 52× impression lift compared with an older stale cohort. The paper also cautions that the 361+ freshness bucket is unstable because it contains only 21 declining pages.

**My methodology question:**  
Does the comparison between refreshed and stale pages support the stronger interpretation that refreshing caused the observed improvement? Pages selected for refreshing may already differ from stale pages in traffic, historical performance, strategic importance, or other characteristics. Without a comparable control group or a stronger pre/post validation design, the observed difference should be interpreted as an association rather than direct evidence of causation.

I would therefore describe the result as directional evidence that freshness is associated with stronger observed performance in this dataset, while avoiding a causal claim that refreshing itself produced the entire measured lift.

---

### Section 1 Takeaway

The main lesson I take from these findings is that a strong numerical difference is not automatically evidence of causation. Before accepting a research claim, I should check how the outcome was defined, what groups were compared, what information was available at the time of measurement, and whether the validation design supports the strength of the conclusion.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Reconstructing the Week-5 modeling dataset

I will first recreate the Week-5 page-level dataset from the FlyRank warehouse on Hugging Face. I will keep the target, features, model configuration, and evaluation metric consistent with Week 5 so that the validation comparison isolates the effect of the split strategy.

In [82]:
!pip install -q datasets

In [83]:
import numpy as np
import pandas as pd

from datasets import load_dataset
from huggingface_hub import login, whoami

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

Connecting to Hugging Face Dataset

In [84]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

login(token=HF_TOKEN)

user_info = whoami()

print("Connected to Hugging Face")
print("Username:", user_info["name"])

Connected to Hugging Face
Username: LEOd2705


In [85]:
dim_content = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_content",
    token=HF_TOKEN
)

fact_query = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_query_90d",
    token=HF_TOKEN
)

print("dim_content:")
print(dim_content)

print("\nfact_content_query_90d:")
print(fact_query)

dim_content:
DatasetDict({
    train: Dataset({
        features: ['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted'],
        num_rows: 519606
    })
})

fact_content_query_90d:
DatasetDict({
    train: Dataset({
        features: ['client_hash_id', 'content_hash_id', 'query_hash_id', 'query_char_count', 'query_token_count', 'window_start', 'window_end', 'impressions_90d', 'clicks_90d', 'impressions_last30', 'clicks_last30', 'impressions_prev30', 'clicks_prev30', 'avg_position_90d', 'avg_position_last30', 'avg_position_prev30', 'content_total_impressions_90d', 'content_visi

In [86]:
dim_content_df = dim_content["train"].to_pandas()
fact_query_df = fact_query["train"].to_pandas()

print("dim_content shape:", dim_content_df.shape)
print("fact_content_query_90d shape:", fact_query_df.shape)

dim_content shape: (519606, 26)
fact_content_query_90d shape: (2414248, 21)


In [87]:
query_agg = (
    fact_query_df
    .groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg(
        impressions_90d=("impressions_90d", "sum"),
        clicks_90d=("clicks_90d", "sum"),
        impressions_last30=("impressions_last30", "sum"),
        clicks_last30=("clicks_last30", "sum"),
        impressions_prev30=("impressions_prev30", "sum"),
        clicks_prev30=("clicks_prev30", "sum"),
        avg_position_90d=("avg_position_90d", "mean"),
        avg_position_last30=("avg_position_last30", "mean"),
        avg_position_prev30=("avg_position_prev30", "mean"),
        content_total_impressions_90d=("content_total_impressions_90d", "first"),
        content_visible_query_count=("content_visible_query_count", "first"),
        rare_query_count=("rare_query_count", "first"),
        rare_impressions_share=("rare_impressions_share", "first"),
        anonymized_impressions_share=("anonymized_impressions_share", "first")
    )
)

print("Aggregated page-level shape:", query_agg.shape)

Aggregated page-level shape: (133852, 16)


In [88]:
df = dim_content_df.merge(
    query_agg,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Final page-level dataset shape:", df.shape)

print(
    "Unique client-content pages:",
    df[["client_hash_id", "content_hash_id"]]
    .drop_duplicates()
    .shape[0]
)

Final page-level dataset shape: (133852, 40)
Unique client-content pages: 133852


In [89]:
df["click_change_pct"] = (
    (df["clicks_last30"] - df["clicks_prev30"])
    / df["clicks_prev30"].replace(0, np.nan)
)

df["is_declining_proxy"] = (
    df["click_change_pct"] <= -0.20
).astype(int)

TARGET = "is_declining_proxy"

print("Target distribution:")
print(df[TARGET].value_counts(dropna=False))

print("\nTarget rate:")
print(f"{df[TARGET].mean():.1%}")

Target distribution:
is_declining_proxy
0    117114
1     16738
Name: count, dtype: int64

Target rate:
12.5%


In [90]:
FEATURES = [
    "impressions_prev30",
    "clicks_prev30",
    "avg_position_prev30",
    "word_count",
    "search_volume",
    "keyword_char_count",
    "url_char_count"
]

print("W05 feature set:")
for feature in FEATURES:
    print("-", feature)

W05 feature set:
- impressions_prev30
- clicks_prev30
- avg_position_prev30
- word_count
- search_volume
- keyword_char_count
- url_char_count


In [91]:
train_df_random, test_df_random = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df[TARGET]
)

print("Random split")
print("Train shape:", train_df_random.shape)
print("Test shape:", test_df_random.shape)

train_clients_random = set(train_df_random["client_hash_id"])
test_clients_random = set(test_df_random["client_hash_id"])

overlap_random = train_clients_random.intersection(test_clients_random)

print(
    "Clients appearing in both train and test:",
    len(overlap_random)
)

Random split
Train shape: (107081, 42)
Test shape: (26771, 42)
Clients appearing in both train and test: 50


In [92]:
X_train_random = train_df_random[FEATURES].copy()
X_test_random = test_df_random[FEATURES].copy()

y_train_random = train_df_random[TARGET]
y_test_random = test_df_random[TARGET]

# Impute using training medians only
for col in FEATURES:
    median_value = X_train_random[col].median()

    if pd.isna(median_value):
        median_value = 0

    X_train_random[col] = X_train_random[col].fillna(median_value)
    X_test_random[col] = X_test_random[col].fillna(median_value)

rf_random = RandomForestClassifier(
    random_state=42,
    n_estimators=100,
    max_depth=6
)

rf_random.fit(X_train_random, y_train_random)

random_prob = rf_random.predict_proba(X_test_random)[:, 1]

print("Random-split Random Forest trained.")

Random-split Random Forest trained.


In [93]:
def precision_at_k(y_true, scores, k):
    temp = pd.DataFrame({
        "y_true": y_true.to_numpy(),
        "score": scores
    })

    top_k = temp.sort_values(
        "score",
        ascending=False
    ).head(k)

    return top_k["y_true"].mean()


K_VALUES = [100, 500, 1000]

random_results = []

for k in K_VALUES:
    random_results.append({
        "K": k,
        "Precision@K": precision_at_k(
            y_test_random,
            random_prob,
            k
        )
    })

random_results_df = pd.DataFrame(random_results)

print("BEFORE — Random Row Split")
display(random_results_df)

BEFORE — Random Row Split


,K,Precision@K
0,100,0.890
1,500,0.822
2,1000,0.790


In [94]:
group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    group_splitter.split(
        df,
        y=df[TARGET],
        groups=df["client_hash_id"]
    )
)

train_df_grouped = df.iloc[train_idx].copy()
test_df_grouped = df.iloc[test_idx].copy()

train_clients_grouped = set(
    train_df_grouped["client_hash_id"]
)

test_clients_grouped = set(
    test_df_grouped["client_hash_id"]
)

overlap_grouped = (
    train_clients_grouped
    .intersection(test_clients_grouped)
)

print("AFTER — Client-Grouped Split")
print("Train shape:", train_df_grouped.shape)
print("Test shape:", test_df_grouped.shape)
print(
    "Clients appearing in both train and test:",
    len(overlap_grouped)
)

AFTER — Client-Grouped Split
Train shape: (117944, 42)
Test shape: (15908, 42)
Clients appearing in both train and test: 0


In [95]:
X_train_grouped = train_df_grouped[FEATURES].copy()
X_test_grouped = test_df_grouped[FEATURES].copy()

y_train_grouped = train_df_grouped[TARGET]
y_test_grouped = test_df_grouped[TARGET]

# Impute using training medians only
for col in FEATURES:
    median_value = X_train_grouped[col].median()

    if pd.isna(median_value):
        median_value = 0

    X_train_grouped[col] = X_train_grouped[col].fillna(median_value)
    X_test_grouped[col] = X_test_grouped[col].fillna(median_value)

rf_grouped = RandomForestClassifier(
    random_state=42,
    n_estimators=100,
    max_depth=6
)

rf_grouped.fit(
    X_train_grouped,
    y_train_grouped
)

grouped_prob = rf_grouped.predict_proba(
    X_test_grouped
)[:, 1]

print("Client-grouped Random Forest trained.")

Client-grouped Random Forest trained.


In [96]:
grouped_results = []

for k in K_VALUES:
    grouped_results.append({
        "K": k,
        "Precision@K": precision_at_k(
            y_test_grouped,
            grouped_prob,
            k
        )
    })

grouped_results_df = pd.DataFrame(grouped_results)

print("AFTER — Client-Grouped Split")
display(grouped_results_df)

AFTER — Client-Grouped Split


,K,Precision@K
0,100,0.790
1,500,0.780
2,1000,0.737


In [97]:
# Failure / Error Examples — Client-Grouped Test Set

error_examples = test_df_grouped.copy()

error_examples["predicted_probability"] = grouped_prob

error_examples["predicted_label"] = (
    error_examples["predicted_probability"] >= 0.50
).astype(int)

# False positives: predicted decline, but actual target is 0
false_positives = (
    error_examples[
        (error_examples[TARGET] == 0) &
        (error_examples["predicted_label"] == 1)
    ]
    .sort_values("predicted_probability", ascending=False)
)

# False negatives: actual decline, but model predicted non-decline
false_negatives = (
    error_examples[
        (error_examples[TARGET] == 1) &
        (error_examples["predicted_label"] == 0)
    ]
    .sort_values("predicted_probability", ascending=True)
)

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\nTop False Positives:")
display(
    false_positives[
        [
            "client_hash_id",
            "content_hash_id",
            TARGET,
            "predicted_probability"
        ]
        + FEATURES
    ].head(5)
)

print("\nTop False Negatives:")
display(
    false_negatives[
        [
            "client_hash_id",
            "content_hash_id",
            TARGET,
            "predicted_probability"
        ]
        + FEATURES
    ].head(5)
)

False positives: 852
False negatives: 3

Top False Positives:


,client_hash_id,content_hash_id,is_declining_proxy,predicted_probability,impressions_prev30,clicks_prev30,avg_position_prev30,word_count,search_volume,keyword_char_count,url_char_count
30745,client_e5c2aa26a8598242,content_3339a52e3b5c2295,0,0.707434,19,1,2.875000,2799.0,10.0,42,100
33594,client_e5c2aa26a8598242,content_c9a78c254bc03250,0,0.701053,5,1,2.750000,3084.0,0.0,31,100
67569,client_e5c2aa26a8598242,content_092f7614ffb1fed4,0,0.699692,26,1,2.895604,2801.0,10.0,36,83
81705,client_fef1a8f436438636,content_a4cddb775af974de,0,0.698739,20,1,44.868421,2604.0,0.0,28,123
32398,client_e5c2aa26a8598242,content_89832667fa63d2aa,0,0.686501,32,2,1.055128,2828.0,10.0,43,117



Top False Negatives:


,client_hash_id,content_hash_id,is_declining_proxy,predicted_probability,impressions_prev30,clicks_prev30,avg_position_prev30,word_count,search_volume,keyword_char_count,url_char_count
130229,client_fef1a8f436438636,content_0aaa197051f58d6f,1,0.470945,29445,10,43.592291,988.0,10.0,42,85
132828,client_fef1a8f436438636,content_6e859edd5208e2f3,1,0.487231,2620,1,9.225508,1064.0,480.0,29,125
82732,client_fef1a8f436438636,content_cc77c45635a231d3,1,0.489687,858,1,3.716658,1054.0,10.0,32,112


### Failure / Error Examples

To complement the aggregate Precision@K results, I inspected individual prediction errors from the client-grouped test set.

The classification threshold used for this diagnostic was 0.50.

The model produced:

- 852 false positives
- 3 false negatives

A false positive is a page that the model classified as declining while the observed `is_declining_proxy` target was 0. A false negative is a page where the observed target was 1 but the model's predicted probability was below the 0.50 threshold.

### Example false positives

Several high-confidence false positives were observed. For example:

| Client | Content | Actual target | Predicted probability |
|---|---|---:|---:|
| `client_e5c2aa26a8598242` | `content_3339a52e3b5c2295` | 0 | 0.707 |
| `client_e5c2aa26a8598242` | `content_c9a78c254bc03250` | 0 | 0.701 |
| `client_e5c2aa26a8598242` | `content_092f7614ffb1fed4` | 0 | 0.700 |
| `client_fef1a8f436438636` | `content_a4cddb775af974de` | 0 | 0.699 |
| `client_e5c2aa26a8598242` | `content_89832667fa63d2aa` | 0 | 0.687 |

These examples show that the model can assign relatively high decline probabilities to pages that do not meet the observed decline-proxy definition. This indicates that the model's ranking signal is not perfectly aligned with the proxy target for every individual page.

### Example false negatives

Only three false negatives were observed:

| Client | Content | Actual target | Predicted probability |
|---|---|---:|---:|
| `client_fef1a8f436438636` | `content_0aaa197051f58d6f` | 1 | 0.471 |
| `client_fef1a8f436438636` | `content_6e859edd5208e2f3` | 1 | 0.487 |
| `client_fef1a8f436438636` | `content_cc77c45635a231d3` | 1 | 0.490 |

The false negatives were close to the 0.50 threshold, suggesting that these examples were not assigned extremely low probabilities. However, this observation is descriptive and does not establish why the model missed them.

### Interpretation

These examples demonstrate why aggregate metrics should be supplemented with individual error inspection. Precision@K measures the quality of the highest-ranked pages, while threshold-based false-positive and false-negative counts describe classification behavior at a chosen probability threshold.

The observed errors show that the model provides a directional ranking signal rather than a perfect determination of whether a page will satisfy the decline proxy. Individual predictions should therefore be treated as decision-support signals and not as definitive classifications.

In [98]:
comparison = random_results_df.merge(
    grouped_results_df,
    on="K",
    suffixes=("_Random_Row", "_Client_Grouped")
)

comparison["Change"] = (
    comparison["Precision@K_Client_Grouped"]
    - comparison["Precision@K_Random_Row"]
)

display(comparison)

,K,Precision@K_Random_Row,Precision@K_Client_Grouped,Change
0,100,0.890,0.790,-0.100
1,500,0.822,0.780,-0.042
2,1000,0.790,0.737,-0.053


### Before vs. After Validation Comparison

The same Random Forest model was evaluated using the same feature set, target definition, model configuration, and Precision@K metric. The validation strategy was the only intended change.

| K | Random Row Split | Client-Grouped Split | Change |
|---:|---:|---:|---:|
| 100 | 0.890 | 0.790 | -0.100 |
| 500 | 0.822 | 0.780 | -0.042 |
| 1000 | 0.790 | 0.737 | -0.053 |

The client-grouped split produced lower Precision@K at all three values of K. At K=100, Precision@K decreased from 0.890 under the random row split to 0.790 under the client-grouped split, a difference of 0.100.

The random row split allows pages belonging to the same client to appear in both training and test sets. The client-grouped split prevents this by holding out complete clients from the test set.

This result suggests that the random row split may provide a more optimistic estimate of model performance for this dataset. The client-grouped result is therefore treated as the more conservative validation estimate for the intended use case of prioritizing content across clients not represented in training.

This comparison should be interpreted as an observed difference between two validation designs, rather than proof that the random split is always optimistic or that the client-grouped result represents performance on all future clients.

### Split Integrity

| Validation design | Client overlap between train and test |
|---|---:|
| Random row split | 50 |
| Client-grouped split | 0 |

The grouped split prevents the same client from appearing in both training and test data, whereas the random row split allowed 50 clients to appear in both.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [99]:
# Section 3 — Leakage Audit

leakage_audit_features = FEATURES + [TARGET]

print("Features and target used in the Week-5 model:\n")

for col in leakage_audit_features:
    print(f"- {col}")

Features and target used in the Week-5 model:

- impressions_prev30
- clicks_prev30
- avg_position_prev30
- word_count
- search_volume
- keyword_char_count
- url_char_count
- is_declining_proxy


In [100]:
print("Target construction:")
print(
    "click_change_pct = "
    "(clicks_last30 - clicks_prev30) / clicks_prev30"
)
print(
    "is_declining_proxy = 1 when click_change_pct <= -0.20"
)

Target construction:
click_change_pct = (clicks_last30 - clicks_prev30) / clicks_prev30
is_declining_proxy = 1 when click_change_pct <= -0.20


In [101]:
# Inspect the time-related and feature columns used by the model

audit_columns = [
    "impressions_prev30",
    "clicks_prev30",
    "avg_position_prev30",
    "impressions_last30",
    "clicks_last30",
    "avg_position_last30",
    "word_count",
    "search_volume",
    "keyword_char_count",
    "url_char_count"
]

available_audit_columns = [
    col for col in audit_columns
    if col in df.columns
]

df[available_audit_columns].head()

,impressions_prev30,clicks_prev30,avg_position_prev30,impressions_last30,clicks_last30,avg_position_last30,word_count,search_volume,keyword_char_count,url_char_count
0,0,0,NaN,16,0,15.562500,2290.0,30.0,32,101
1,139,0,30.932800,123,0,26.543960,2087.0,0.0,36,100
2,59,0,34.091049,38,0,34.201028,2567.0,0.0,22,96
3,0,0,NaN,28,0,72.844920,2449.0,20.0,31,102
4,9,0,63.555556,6,0,80.666667,2362.0,0.0,31,99


In [102]:
print("Feature availability / missingness check:\n")

for col in FEATURES:
    print(
        f"{col}: "
        f"missing={df[col].isna().sum():,}, "
        f"missing_rate={df[col].isna().mean():.2%}"
    )

Feature availability / missingness check:

impressions_prev30: missing=0, missing_rate=0.00%
clicks_prev30: missing=0, missing_rate=0.00%
avg_position_prev30: missing=13,666, missing_rate=10.21%
word_count: missing=33,297, missing_rate=24.88%
search_volume: missing=2,289, missing_rate=1.71%
keyword_char_count: missing=0, missing_rate=0.00%
url_char_count: missing=0, missing_rate=0.00%


In [103]:
# Inspect the source columns used to construct the model features

feature_source_columns = [
    "word_count",
    "search_volume",
    "keyword_char_count",
    "url_char_count"
]

for col in feature_source_columns:
    print(f"\n{col}")
    print("dtype:", df[col].dtype)
    print("unique values:", df[col].nunique(dropna=True))
    print("sample values:", df[col].dropna().head(5).tolist())


word_count
dtype: float64
unique values: 6199
sample values: [2290.0, 2087.0, 2567.0, 2449.0, 2362.0]

search_volume
dtype: float64
unique values: 47
sample values: [30.0, 0.0, 0.0, 20.0, 0.0]

keyword_char_count
dtype: int64
unique values: 73
sample values: [32, 36, 22, 31, 31]

url_char_count
dtype: int64
unique values: 152
sample values: [101, 100, 96, 102, 99]


In [104]:
# Check whether any model feature is directly derived from the outcome-period columns

outcome_columns = [
    "clicks_last30",
    "impressions_last30",
    "avg_position_last30"
]

print("Outcome-period columns:")
for col in outcome_columns:
    print("-", col)

print("\nModel features:")
for col in FEATURES:
    print("-", col)

print(
    "\nDirect overlap between outcome-period columns and model features:",
    set(outcome_columns).intersection(FEATURES)
)

Outcome-period columns:
- clicks_last30
- impressions_last30
- avg_position_last30

Model features:
- impressions_prev30
- clicks_prev30
- avg_position_prev30
- word_count
- search_volume
- keyword_char_count
- url_char_count

Direct overlap between outcome-period columns and model features: set()


In [105]:
# Compare feature-target relationships

feature_target_corr = (
    df[FEATURES + [TARGET]]
    .corr(numeric_only=True)[TARGET]
    .drop(TARGET)
    .sort_values(key=abs, ascending=False)
)

print("Feature correlation with target:")
display(feature_target_corr.to_frame("correlation_with_target"))

Feature correlation with target:


,correlation_with_target
avg_position_prev30,-0.252494
clicks_prev30,0.219796
impressions_prev30,0.200757
keyword_char_count,0.023046
word_count,-0.011572
search_volume,-0.011569
url_char_count,-0.008988


In [106]:
# Inspect the feature columns alongside the variables used to construct the target

audit_summary = pd.DataFrame({
    "feature": FEATURES,
    "used_directly_in_target_formula": [
        feature in ["clicks_last30", "clicks_prev30"]
        for feature in FEATURES
    ]
})

display(audit_summary)

,feature,used_directly_in_target_formula
0,impressions_prev30,False
1,clicks_prev30,True
2,avg_position_prev30,False
3,word_count,False
4,search_volume,False
5,keyword_char_count,False
6,url_char_count,False


## Leakage Audit

The Week-5 model uses seven features to predict the `is_declining_proxy` target. The target is constructed from the change in clicks between the previous 30-day period and the last 30-day period:

`click_change_pct = (clicks_last30 - clicks_prev30) / clicks_prev30`

A feature was considered higher risk if it directly used outcome-period information or was constructed using information that would only become available after the prediction point.

### Feature-level audit

| Feature | Relationship to prediction period | Leakage assessment |
|---|---|---|
| `impressions_prev30` | Historical impressions from the previous 30-day period | Low leakage risk |
| `clicks_prev30` | Historical clicks from the previous 30-day period and denominator in the target calculation | Low leakage risk, with timing dependency |
| `avg_position_prev30` | Historical average position from the previous 30-day period | Low leakage risk |
| `word_count` | Content-length characteristic | Requires timing assumption |
| `search_volume` | Search-demand characteristic | Requires timing assumption |
| `keyword_char_count` | Derived from keyword/query text | Low leakage risk if query text is available before prediction |
| `url_char_count` | Derived from the URL | Low leakage risk |

### Direct outcome-period overlap check

The outcome-period variables inspected were:

- `clicks_last30`
- `impressions_last30`
- `avg_position_last30`

The direct overlap between these outcome-period variables and the seven model features was an empty set.

Therefore, none of the Week-5 model features directly uses the inspected outcome-period columns.

### Target-formula check

`clicks_prev30` appears in the target formula because it is used as the historical comparison period:

`(clicks_last30 - clicks_prev30) / clicks_prev30`

This does not by itself constitute leakage because `clicks_prev30` represents information from the previous period rather than the outcome period. However, this conclusion depends on the prediction point being defined after the previous 30-day period and before the last 30-day outcome period.

### Feature-target correlation diagnostic

The observed correlations with the target were:

| Feature | Correlation with `is_declining_proxy` |
|---|---:|
| `avg_position_prev30` | -0.252494 |
| `clicks_prev30` | 0.219796 |
| `impressions_prev30` | 0.200757 |
| `keyword_char_count` | 0.023046 |
| `word_count` | -0.011572 |
| `search_volume` | -0.011569 |
| `url_char_count` | -0.008988 |

These correlations are treated only as diagnostics. A feature having a relationship with the target does not by itself indicate leakage. In particular, the stronger relationships observed for historical position, clicks, and impressions can be legitimate predictive relationships.

### Leakage conclusion

Based on the checks performed, I found no evidence that the Week-5 model directly includes the inspected outcome-period variables as features. The historical `prev30` variables are consistent with a prediction setup in which the previous 30 days are available before the outcome period.

However, the audit cannot establish that every feature is completely leakage-free without confirming the exact timestamp or snapshot used to create features such as `word_count` and `search_volume`. Therefore, I treat these as timing assumptions rather than making a stronger claim that the entire feature set is definitively leakage-free.

The main validation lesson is that leakage auditing requires checking both **what information a feature contains** and **when that information would have been available**, rather than relying only on feature names or correlations.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

The Week-5 model was developed as a prioritization model for identifying content pages that may warrant review based on observable search and content-performance signals.

After applying the validation and leakage audit, some claims should be narrowed so that they match the evidence more closely.

### Claim 1 — Model performance

**Original / stronger claim:**

> The model achieves high Precision@K and can accurately identify declining pages.

**Why this needs to be revised:**

The random-row validation produced Precision@100 of 0.890, but the client-grouped validation produced Precision@100 of 0.790. The same decrease was observed at K=500 and K=1000.

Therefore, the random-row result should not be presented as the only estimate of model performance or as proof of generalization to unseen clients.

**Rewritten claim:**

> Under the client-grouped validation used in this audit, the model achieved a measured Precision@100 of 0.790, with Precision@500 of 0.780 and Precision@1000 of 0.737. These results indicate directional prioritization performance on the evaluated dataset rather than guaranteed performance on future clients.

---

### Claim 2 — Generalization

**Original / stronger claim:**

> The model generalizes well across clients.

**Why this needs to be revised:**

The client-grouped split prevents any client from appearing in both training and test data, making it a more conservative test of cross-client generalization. However, the experiment evaluates only the available dataset and one grouped split.

**Rewritten claim:**

> The client-grouped evaluation provides an observed estimate of model performance when test clients are held out from training. The measured Precision@K values suggest that the model retains useful directional signal under this stricter validation design, although broader claims about generalization would require evaluation across additional clients and validation periods.

---

### Claim 3 — Leakage

**Original / stronger claim:**

> The model has no data leakage.

**Why this needs to be revised:**

The audit found no direct overlap between the model features and the inspected outcome-period variables. However, the exact timing of some features, particularly `word_count` and `search_volume`, was not independently verified.

**Rewritten claim:**

> The leakage audit found no direct use of the inspected outcome-period variables in the Week-5 feature set. Historical `prev30` features are consistent with the intended prediction boundary, while the timing of some static or contextual features remains an assumption that should be verified before making a stronger leakage-free claim.

---

### Claim 4 — Business impact

**Original / stronger claim:**

> The model identifies pages that should be refreshed first and will improve SEO performance.

**Why this needs to be revised:**

The target is a proxy for historical click decline rather than a directly measured business outcome. The evaluation measures Precision@K, not the downstream effect of actually refreshing pages.

**Rewritten claim:**

> The model can be evaluated as a decision-support tool for prioritizing pages associated with the observed decline proxy. Whether acting on these recommendations improves traffic, CTR, or other business outcomes was not measured in this experiment.

---

### Final public-safe claim

Based on the validation and leakage audit, the strongest supported summary is:

> The Week-5 model provides a measured directional ranking signal for prioritizing content pages associated with an observed decline proxy. Under a client-grouped validation split, it achieved Precision@100 of 0.790, Precision@500 of 0.780, and Precision@1000 of 0.737 on the evaluated dataset. The grouped results were lower than the corresponding random-row results, indicating that validation design materially affects the measured performance. The results should therefore be interpreted as decision-support evidence on the evaluated data rather than as a guarantee of future performance or business impact.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.